In [1]:
# Cell 1 — bootstrap + imports
import sys
import re
import json
import random
from pathlib import Path
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fineas.io.paths import norm_ohlcv_part_path, features_daily_part_path
from fineas.io.qc import qc_norm_ohlcv_partition, qc_features_partition
from fineas.features.technicals import V1_FEATURE_COLS

In [2]:
# Cell 2 — parameterized partition selector (explicit OR random existing)
INTERVAL = "1d"
USE_RANDOM_EXISTING = True
FORCE_NON_JAN = True     # set False if you want any month
SEED = 42

# manual fallback if USE_RANDOM_EXISTING = False
SYMBOL = "AAPL"
YEAR = 2022
MONTH = 1

random.seed(SEED)

feat_interval_root = ROOT / "data" / "norm" / "features_daily" / f"interval={INTERVAL}"

def _parse_symbol_year_month(p: Path):
    s = str(p).replace("\\", "/")
    m = re.search(r"symbol=([^/]+)/year=(\d{4})/month=(\d{2})/part-(\d{4})-(\d{2})\.parquet$", s)
    if not m:
        raise ValueError(f"could not parse partition from path: {p}")
    sym = m.group(1)
    y = int(m.group(2))
    mo = int(m.group(3))
    return sym, y, mo

if USE_RANDOM_EXISTING:
    feat_parts = sorted(feat_interval_root.rglob("part-*.parquet"))
    assert feat_parts, f"no feature partitions found under: {feat_interval_root}"

    candidates = feat_parts
    if FORCE_NON_JAN:
        candidates = [p for p in feat_parts if _parse_symbol_year_month(p)[2] != 1]
        assert candidates, "no non-January feature partitions found"

    pick = random.choice(candidates)
    SYMBOL, YEAR, MONTH = _parse_symbol_year_month(pick)

norm_p = norm_ohlcv_part_path(SYMBOL, INTERVAL, YEAR, MONTH, repo_root=ROOT)
feat_p = features_daily_part_path(SYMBOL, INTERVAL, YEAR, MONTH, repo_root=ROOT)

print("Selected:", {"symbol": SYMBOL, "year": YEAR, "month": MONTH, "interval": INTERVAL})
print("norm_p:", norm_p)
print("feat_p:", feat_p)

Selected: {'symbol': 'NVDA', 'year': 2023, 'month': 6, 'interval': '1d'}
norm_p: C:\Users\quantbase\Desktop\fineas\data\norm\ohlcv\interval=1d\symbol=NVDA\year=2023\month=06\part-2023-06.parquet
feat_p: C:\Users\quantbase\Desktop\fineas\data\norm\features_daily\interval=1d\symbol=NVDA\year=2023\month=06\part-2023-06.parquet


In [3]:
# Cell 3 — load paired partitions
assert norm_p.exists(), f"missing norm partition: {norm_p}"
assert feat_p.exists(), f"missing feature partition: {feat_p}"

norm_df = pd.read_parquet(norm_p)
feat_df = pd.read_parquet(feat_p)

print("norm shape:", norm_df.shape)
print("feat shape:", feat_df.shape)
print("norm cols:", norm_df.columns.tolist())
print("feat cols:", feat_df.columns.tolist())

norm shape: (21, 10)
feat shape: (21, 20)
norm cols: ['ts', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'returns', 'movement']
feat cols: ['ts', 'symbol', 'open', 'high', 'low', 'close', 'adj_close', 'volume', 'returns', 'movement', 'log_ret_1', 'ret_2', 'ret_5', 'range_pct', 'body_pct', 'close_pos_in_range', 'volume_chg_1', 'volatility_5', 'sma_5_ratio', 'sma_10_ratio']


In [4]:
#  QC gates (norm + features)
qc_norm = qc_norm_ohlcv_partition(norm_df, name=f"qc_norm:{SYMBOL}/{YEAR}-{MONTH:02d}")
qc_feat = qc_features_partition(feat_df, name=f"qc_feat:{SYMBOL}/{YEAR}-{MONTH:02d}", expected_feature_cols=V1_FEATURE_COLS)

print("qc_norm ok:", qc_norm.ok)
print("qc_feat ok:", qc_feat.ok)
display(qc_norm.to_dict())
display(qc_feat.to_dict())

assert qc_norm.ok, qc_norm.to_dict()
assert qc_feat.ok, qc_feat.to_dict()

qc_norm ok: True
qc_feat ok: True


{'ok': True,
 'hard_fails': [],
 'warnings': ['qc_norm:NVDA/2023-06: returns present where intra-part validation is unavailable (likely cross-part lookback) (n=1)'],
 'stats': {'name': 'qc_norm:NVDA/2023-06',
  'rows': 21,
  'key_name': 'qc_norm:NVDA/2023-06/key',
  'key_rows': 21,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'qc_norm:NVDA/2023-06/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'ohlcv_name': 'qc_norm:NVDA/2023-06/ohlcv',
  'ohlcv_rows': 21,
  'ohlcv_ohlc_violation_rows': 0,
  'ohlcv_volume_negative_rows': 0,
  'ohlcv_nan_counts': {'open': 0,
   'high': 0,
   'low': 0,
   'close': 0,
   'volume': 0},
  'returns_mismatch_n': 0,
  'returns_missing_expected_n': 0,
  'movement_mismatch_n': 0,
  'cross_part_unverifiable_returns_n': 1}}

{'ok': True,
 'hard_fails': [],
 'warnings': [],
 'stats': {'name': 'qc_feat:NVDA/2023-06',
  'rows': 21,
  'key_name': 'qc_feat:NVDA/2023-06/key',
  'key_rows': 21,
  'key_key_cols': ['symbol', 'ts'],
  'key_dup_keys_n': 0,
  'time_name': 'qc_feat:NVDA/2023-06/time',
  'time_monotonic_bad_symbols': [],
  'time_max_gap_days': 4.0,
  'time_gap_warn_symbols': [],
  'movement_invalid_values': [],
  'movement_present_where_returns_nan_n': 0,
  'feature_cols': ['log_ret_1',
   'ret_2',
   'ret_5',
   'range_pct',
   'body_pct',
   'close_pos_in_range',
   'volume_chg_1',
   'volatility_5',
   'sma_5_ratio',
   'sma_10_ratio'],
  'feature_cols_n': 10,
  'feature_non_numeric_cols': [],
  'feature_inf_counts': {'log_ret_1': 0,
   'ret_2': 0,
   'ret_5': 0,
   'range_pct': 0,
   'body_pct': 0,
   'close_pos_in_range': 0,
   'volume_chg_1': 0,
   'volatility_5': 0,
   'sma_5_ratio': 0,
   'sma_10_ratio': 0},
  'feature_nan_counts': {'log_ret_1': 0,
   'ret_2': 0,
   'ret_5': 0,
   'range_pct': 0

In [5]:
# key-set equality (norm vs features)
a = norm_df[["symbol", "ts"]].copy()
b = feat_df[["symbol", "ts"]].copy()

a["ts"] = pd.to_datetime(a["ts"], utc=True)
b["ts"] = pd.to_datetime(b["ts"], utc=True)

a_set = set(map(tuple, a.to_numpy()))
b_set = set(map(tuple, b.to_numpy()))

missing_in_feat = sorted(list(a_set - b_set))[:10]
extra_in_feat = sorted(list(b_set - a_set))[:10]

assert a_set == b_set, {
    "missing_in_feat_sample": missing_in_feat,
    "extra_in_feat_sample": extra_in_feat,
    "norm_keys_n": len(a_set),
    "feat_keys_n": len(b_set),
}

print("OK: norm -> features key sets match exactly.")

OK: norm -> features key sets match exactly.


In [6]:
# Cell 6 — carry-through column equality checks
n = norm_df.copy()
f = feat_df.copy()

n["ts"] = pd.to_datetime(n["ts"], utc=True)
f["ts"] = pd.to_datetime(f["ts"], utc=True)

m = n.merge(f, on=["symbol", "ts"], suffixes=("_norm", "_feat"), how="inner", validate="one_to_one")

float_cols = ["open", "high", "low", "close", "adj_close", "returns"]
int_like_cols = ["volume"]

for c in float_cols:
    x = pd.to_numeric(m[f"{c}_norm"], errors="coerce").to_numpy()
    y = pd.to_numeric(m[f"{c}_feat"], errors="coerce").to_numpy()
    assert np.allclose(x, y, equal_nan=True, rtol=0.0, atol=1e-12), f"{c} mismatch"

for c in int_like_cols:
    x = pd.to_numeric(m[f"{c}_norm"], errors="coerce").to_numpy()
    y = pd.to_numeric(m[f"{c}_feat"], errors="coerce").to_numpy()
    assert np.array_equal(x, y) or np.allclose(x, y, equal_nan=True), f"{c} mismatch"

# movement equality with NaN-safe comparison
sentinel = "__NA__"
mv_norm = m["movement_norm"].astype("object").where(m["movement_norm"].notna(), sentinel).astype(str)
mv_feat = m["movement_feat"].astype("object").where(m["movement_feat"].notna(), sentinel).astype(str)
assert (mv_norm == mv_feat).all(), "movement mismatch"

print("OK: carry-through columns are identical (norm == features).")

OK: carry-through columns are identical (norm == features).


In [7]:
# Cell 7 — recompute subset of features (exactness checks)
d = feat_df.copy().sort_values(["symbol", "ts"], kind="mergesort").reset_index(drop=True)

# 1) range_pct
exp_range_pct = ((d["high"] - d["low"]) / d["close"]) * 100.0
mask = exp_range_pct.notna() & d["range_pct"].notna()
assert float((d.loc[mask, "range_pct"] - exp_range_pct[mask]).abs().max()) <= 1e-12

# 2) body_pct
exp_body_pct = ((d["close"] - d["open"]) / d["open"]) * 100.0
mask = exp_body_pct.notna() & d["body_pct"].notna()
assert float((d.loc[mask, "body_pct"] - exp_body_pct[mask]).abs().max()) <= 1e-12

# 3) close_pos_in_range (zero-range guarded)
bar_range = d["high"] - d["low"]
exp_close_pos = pd.Series(np.nan, index=d.index, dtype="float64")
m0 = bar_range.notna() & (bar_range != 0) & d["close"].notna() & d["low"].notna()
exp_close_pos.loc[m0] = (d.loc[m0, "close"] - d.loc[m0, "low"]) / bar_range.loc[m0]
mask = exp_close_pos.notna() & d["close_pos_in_range"].notna()
assert float((d.loc[mask, "close_pos_in_range"] - exp_close_pos[mask]).abs().max()) <= 1e-12

# 4) rolling checks within-partition (exact where local recompute is available)
exp_sma5 = d.groupby("symbol")["adj_close"].transform(lambda s: s.rolling(5, min_periods=5).mean())
exp_sma5_ratio = d["adj_close"] / exp_sma5
mask = exp_sma5_ratio.notna() & d["sma_5_ratio"].notna()
assert float((d.loc[mask, "sma_5_ratio"] - exp_sma5_ratio[mask]).abs().max()) <= 1e-12

exp_vol5 = d.groupby("symbol")["returns"].transform(lambda s: s.rolling(5, min_periods=5).std())
mask = exp_vol5.notna() & d["volatility_5"].notna()
assert float((d.loc[mask, "volatility_5"] - exp_vol5[mask]).abs().max()) <= 1e-12

print("Phase 3E recomputation checks: PASS")

Phase 3E recomputation checks: PASS


In [8]:
# Cell 8 — warmup NaN profile inspection
qc_feat_dict = qc_feat.to_dict()
nan_counts = qc_feat_dict["stats"]["feature_nan_counts"]
nan_row = pd.DataFrame([nan_counts], index=[f"{SYMBOL}-{YEAR}-{MONTH:02d}"])
display(nan_row)

# sanity expectations for instantaneous bar-shape features (assuming OHLC exists)
for c in ["range_pct", "body_pct", "close_pos_in_range"]:
    assert nan_counts.get(c, 0) == 0, {c: nan_counts.get(c, None)}

print("OK: warmup NaNs are explicit and bounded; instantaneous features have no NaN warmup.")

,log_ret_1,ret_2,ret_5,range_pct,body_pct,close_pos_in_range,volume_chg_1,volatility_5,sma_5_ratio,sma_10_ratio
NVDA-2023-06,0,0,0,0,0,0,0,0,0,0


OK: warmup NaNs are explicit and bounded; instantaneous features have no NaN warmup.


In [9]:
# Cell 9 — prior-tail continuity probe (non-January month preferred)
# Goal: for non-Jan months, early rows of rolling features should often be populated due to prior-tail usage.
x = feat_df.copy().sort_values(["symbol", "ts"]).reset_index(drop=True)

print(f"Selected month: {YEAR}-{MONTH:02d}")
display(x[["ts", "ret_5", "volatility_5", "sma_5_ratio", "sma_10_ratio"]].head(12))

if MONTH != 1:
    # These should generally not be all-NaN at the month start if prior-tail continuity worked.
    # We use "at least one non-NaN" to avoid over-constraining edge cases.
    assert x["sma_5_ratio"].head(4).notna().any(), "sma_5_ratio looks like no prior-tail continuity"
    assert x["sma_10_ratio"].head(9).notna().any(), "sma_10_ratio looks like no prior-tail continuity"
    print("OK: prior-tail continuity appears active (non-January month).")
else:
    print("January partition selected; continuity probe skipped (expected warmup NaNs at month start).")

Selected month: 2023-06


,ts,ret_5,volatility_5,sma_5_ratio,sma_10_ratio
0,2023-06-01 00:00:00+00:00,30.231198,11.129741,1.021624,1.136334
1,2023-06-02 00:00:00+00:00,3.546606,4.245195,1.003301,1.099644
2,2023-06-05 00:00:00+00:00,0.577729,4.140866,0.998175,1.071590
3,2023-06-06 00:00:00+00:00,-3.632442,3.848771,0.992370,1.036247
4,2023-06-07 00:00:00+00:00,-0.938634,3.100552,0.963958,0.986780
5,2023-06-08 00:00:00+00:00,-3.158193,2.125885,0.997023,0.993177
6,2023-06-09 00:00:00+00:00,-1.406127,2.172389,1.006637,0.997839
7,2023-06-12 00:00:00+00:00,0.804373,2.359751,1.023449,1.014753
8,2023-06-13 00:00:00+00:00,6.137143,2.664009,1.050451,1.051860
9,2023-06-14 00:00:00+00:00,14.735171,1.636622,1.070744,1.088086


OK: prior-tail continuity appears active (non-January month).


In [10]:
# Cell 10 — features_report.json cross-check (Phase 3D acceptance)
runs_dir = ROOT / "data" / "meta" / "runs"
latest_run = sorted([p for p in runs_dir.glob("run=*") if p.is_dir()])[-1]
report_path = latest_run / "features_report.json"

payload = json.loads(report_path.read_text(encoding="utf-8"))
summary = payload["summary"]

display(summary)

assert summary["expected_parts"] == summary["ok_parts"], summary
assert summary["missing_norm_parts"] == 0, summary
assert summary["empty_norm_parts"] == 0, summary
assert summary["missing_cols_norm_parts"] == 0, summary
assert summary["qc_failed_parts"] == 0, summary
assert summary["error_parts"] == 0, summary
assert len(payload["results"]) == summary["expected_parts"], {
    "results_len": len(payload["results"]),
    "expected_parts": summary["expected_parts"],
}

print("OK: Phase 3D report acceptance checks pass.")

{'run_id': 'run=20260304T102000Z',
 'created_utc': '2026-03-05T05:45:30.481963+00:00',
 'interval': '1d',
 'start': '2022-01-01',
 'end': '2024-01-01',
 'symbols': ['AAPL', 'MSFT', 'NVDA', 'NEWGEN.NS'],
 'months': [[2022, 1],
  [2022, 2],
  [2022, 3],
  [2022, 4],
  [2022, 5],
  [2022, 6],
  [2022, 7],
  [2022, 8],
  [2022, 9],
  [2022, 10],
  [2022, 11],
  [2022, 12],
  [2023, 1],
  [2023, 2],
  [2023, 3],
  [2023, 4],
  [2023, 5],
  [2023, 6],
  [2023, 7],
  [2023, 8],
  [2023, 9],
  [2023, 10],
  [2023, 11],
  [2023, 12]],
 'lookback_rows': 20,
 'expected_parts': 96,
 'ok_parts': 96,
 'missing_norm_parts': 0,
 'empty_norm_parts': 0,
 'missing_cols_norm_parts': 0,
 'qc_failed_parts': 0,
 'error_parts': 0,
 'feature_cols': ['log_ret_1',
  'ret_2',
  'ret_5',
  'range_pct',
  'body_pct',
  'close_pos_in_range',
  'volume_chg_1',
  'volatility_5',
  'sma_5_ratio',
  'sma_10_ratio'],
 'feature_warn_partitions': 4,
 'prev_tail_used_parts': 92,
 'feature_nan_counts_by_symbol': {'AAPL': {'l

OK: Phase 3D report acceptance checks pass.


In [11]:
# Cell 11 — diagnostics table + closeout verdict
status_counts = Counter([r.get("status") for r in payload["results"]])
print("status_counts:", status_counts)

nan_tbl = pd.DataFrame(summary["feature_nan_counts_by_symbol"]).T
display(nan_tbl)

print("PHASE 3 CLOSEOUT: PASS")

status_counts: Counter({'ok': 96})


,log_ret_1,ret_2,ret_5,range_pct,body_pct,close_pos_in_range,volume_chg_1,volatility_5,sma_5_ratio,sma_10_ratio
AAPL,1,2,5,0,0,0,1,5,4,9
MSFT,1,2,5,0,0,0,1,5,4,9
NVDA,1,2,5,0,0,0,1,5,4,9
NEWGEN.NS,1,2,5,0,0,0,1,5,4,9


PHASE 3 CLOSEOUT: PASS
